In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import joblib


In [8]:
from pathlib import Path

# Find the project root (the parent of the notebooks folder)
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data" / "cleaned"

accounts = pd.read_csv(data_dir / "ravenstack_accounts_clean.csv")
subscriptions = pd.read_csv(data_dir / "ravenstack_subscriptions_clean.csv")
feature_usage = pd.read_csv(data_dir / "ravenstack_feature_usage_clean.csv")
support_tickets = pd.read_csv(data_dir / "ravenstack_support_tickets_clean.csv")
churn_events = pd.read_csv(data_dir / "ravenstack_churn_events_clean.csv")
account_data = pd.read_csv(data_dir / "account_data.csv")

print("All datasets loaded successfully!")
print("Accounts:", accounts.shape)
print("Subscriptions:", subscriptions.shape)
print("Feature usage:", feature_usage.shape)
print("Support tickets:", support_tickets.shape)
print("Churn events:", churn_events.shape)
print("Account data:", account_data.shape)


for df, columns in [
    (accounts, ["signup_date"]),
    (subscriptions, ["start_date", "end_date"]),
    (feature_usage, ["usage_date"]),
    (support_tickets, ["submitted_at", "closed_at"]),
    (churn_events, ["churn_date"])
]:
    for column in columns:
        df[column] = pd.to_datetime(df[column])

print("Datasets loaded successfully.")


print("dataset loaded")

All datasets loaded successfully!
Accounts: (500, 10)
Subscriptions: (5000, 14)
Feature usage: (25000, 8)
Support tickets: (2000, 9)
Churn events: (600, 9)
Account data: (500, 33)
Datasets loaded successfully.
dataset loaded


In [9]:
# Latest date available in the dataset
observation_end = max(
    feature_usage["usage_date"].max(),
    support_tickets["submitted_at"].max(),
    subscriptions["end_date"].dropna().max(),
    churn_events["churn_date"].max()
)

# First churn date only for customers whose final target is churn = 1
churned_accounts = accounts.loc[
    accounts["churn_flag"] == 1,
    "account_id"
]

account_churn_date = (
    churn_events[
        churn_events["account_id"].isin(churned_accounts)
    ]
    .groupby("account_id")["churn_date"]
    .min()
)

accounts["churn_date"] = accounts["account_id"].map(
    account_churn_date
)

# Default: use the end of the observation period
accounts["prediction_date"] = pd.Timestamp(observation_end)

# For actual churners: predict 30 days before churn
churned_mask = accounts["churn_flag"] == 1

accounts.loc[churned_mask, "prediction_date"] = (
    accounts.loc[churned_mask, "churn_date"]
    - pd.Timedelta(days=30)
)

print("Observation end:", observation_end)
print("\nPrediction dates created successfully.")

print("\nPrediction date examples:")
print(
    accounts[
        ["account_id", "churn_flag", "churn_date", "prediction_date"]
    ].head()
)

Observation end: 2024-12-31 00:00:00

Prediction dates created successfully.

Prediction date examples:
  account_id  churn_flag churn_date prediction_date
0   A-2e4581           0        NaT      2024-12-31
1   A-43a9e3           1        NaT             NaT
2   A-0a282f           0        NaT      2024-12-31
3   A-1f0ac7           0        NaT      2024-12-31
4   A-ce550d           1 2024-12-28      2024-11-28


In [10]:
# Basic customer information
model_data = accounts[[
    "account_id",
    "industry",
    "country",
    "plan_tier",
    "seats",
    "is_trial",
    "churn_flag",
    "signup_date",
    "prediction_date"
]].copy()

# Customer tenure at prediction date
model_data["tenure_months"] = (
    model_data["prediction_date"] -
    model_data["signup_date"]
).dt.days / 30.44


# --------------------------------------------------
# 1. Subscription information before prediction date
# --------------------------------------------------

subscription_pre = subscriptions.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

subscription_pre = subscription_pre[
    subscription_pre["start_date"]
    <= subscription_pre["prediction_date"]
]

subscription_summary = (
    subscription_pre
    .groupby("account_id")
    .agg(
        avg_mrr=("mrr_amount", "mean"),
        upgrade_count=("upgrade_flag", "sum"),
        downgrade_count=("downgrade_flag", "sum"),
        auto_renew_count=("auto_renew_flag", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 2. Feature usage before prediction date
# --------------------------------------------------

usage_pre = feature_usage.merge(
    subscriptions[["subscription_id", "account_id"]],
    on="subscription_id",
    how="left"
)

usage_pre = usage_pre.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

usage_pre = usage_pre[
    usage_pre["usage_date"]
    <= usage_pre["prediction_date"]
]

usage_summary = (
    usage_pre
    .groupby("account_id")
    .agg(
        total_usage=("usage_count", "sum"),
        total_usage_duration=("usage_duration_secs", "sum"),
        total_errors=("error_count", "sum"),
        beta_feature_usage=("is_beta_feature", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 3. Support information before prediction date
# --------------------------------------------------

support_pre = support_tickets.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

support_pre = support_pre[
    support_pre["submitted_at"]
    <= support_pre["prediction_date"]
]

support_summary = (
    support_pre
    .groupby("account_id")
    .agg(
        total_tickets=("ticket_id", "count"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        avg_response_minutes=("first_response_time_minutes", "mean"),
        avg_satisfaction=("satisfaction_score", "mean"),
        escalated_tickets=("escalation_flag", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 4. Combine all information
# --------------------------------------------------

model_data = (
    model_data
    .merge(
        subscription_summary,
        on="account_id",
        how="left"
    )
    .merge(
        usage_summary,
        on="account_id",
        how="left"
    )
    .merge(
        support_summary,
        on="account_id",
        how="left"
    )
)

print("Model dataset shape:", model_data.shape)
model_data.head()

Model dataset shape: (500, 23)


,account_id,industry,country,plan_tier,seats,is_trial,churn_flag,signup_date,prediction_date,tenure_months,...,auto_renew_count,total_usage,total_usage_duration,total_errors,beta_feature_usage,total_tickets,avg_resolution_hours,avg_response_minutes,avg_satisfaction,escalated_tickets
0,A-2e4581,EdTech,United States,Basic,9,0,0,2024-10-16,2024-12-31,2.496715,...,8.0,535.0,152339.0,38.0,4.0,2.0,23.000000,91.000000,3.000000,0.0
1,A-43a9e3,FinTech,India,Basic,18,0,1,2023-08-17,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A-0a282f,DevTools,United States,Basic,1,0,0,2024-08-27,2024-12-31,4.139290,...,15.0,821.0,251210.0,48.0,5.0,3.0,43.666667,63.666667,4.666667,0.0
3,A-1f0ac7,HealthTech,United Kingdom,Basic,24,1,0,2023-08-27,2024-12-31,16.162943,...,5.0,382.0,102528.0,21.0,5.0,2.0,29.000000,174.000000,NaN,0.0
4,A-ce550d,HealthTech,United States,Enterprise,35,0,1,2024-10-27,2024-11-28,1.051248,...,3.0,559.0,209127.0,28.0,4.0,7.0,42.285714,107.857143,3.800000,1.0


In [11]:
# Basic customer information
model_data = accounts[[
    "account_id",
    "industry",
    "country",
    "plan_tier",
    "seats",
    "is_trial",
    "churn_flag",
    "signup_date",
    "prediction_date"
]].copy()

# Customer tenure at prediction date
model_data["tenure_months"] = (
    model_data["prediction_date"] -
    model_data["signup_date"]
).dt.days / 30.44


# --------------------------------------------------
# 1. Subscription information before prediction date
# --------------------------------------------------

subscription_pre = subscriptions.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

subscription_pre = subscription_pre[
    subscription_pre["start_date"]
    <= subscription_pre["prediction_date"]
]

subscription_summary = (
    subscription_pre
    .groupby("account_id")
    .agg(
        avg_mrr=("mrr_amount", "mean"),
        upgrade_count=("upgrade_flag", "sum"),
        downgrade_count=("downgrade_flag", "sum"),
        auto_renew_count=("auto_renew_flag", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 2. Feature usage before prediction date
# --------------------------------------------------

usage_pre = feature_usage.merge(
    subscriptions[["subscription_id", "account_id"]],
    on="subscription_id",
    how="left"
)

usage_pre = usage_pre.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

usage_pre = usage_pre[
    usage_pre["usage_date"]
    <= usage_pre["prediction_date"]
]

usage_summary = (
    usage_pre
    .groupby("account_id")
    .agg(
        total_usage=("usage_count", "sum"),
        total_usage_duration=("usage_duration_secs", "sum"),
        total_errors=("error_count", "sum"),
        beta_feature_usage=("is_beta_feature", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 3. Support information before prediction date
# --------------------------------------------------

support_pre = support_tickets.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

support_pre = support_pre[
    support_pre["submitted_at"]
    <= support_pre["prediction_date"]
]

support_summary = (
    support_pre
    .groupby("account_id")
    .agg(
        total_tickets=("ticket_id", "count"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        avg_response_minutes=("first_response_time_minutes", "mean"),
        avg_satisfaction=("satisfaction_score", "mean"),
        escalated_tickets=("escalation_flag", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 4. Combine all information
# --------------------------------------------------

model_data = (
    model_data
    .merge(
        subscription_summary,
        on="account_id",
        how="left"
    )
    .merge(
        usage_summary,
        on="account_id",
        how="left"
    )
    .merge(
        support_summary,
        on="account_id",
        how="left"
    )
)

print("Model dataset shape:", model_data.shape)
model_data.head()

Model dataset shape: (500, 23)


,account_id,industry,country,plan_tier,seats,is_trial,churn_flag,signup_date,prediction_date,tenure_months,...,auto_renew_count,total_usage,total_usage_duration,total_errors,beta_feature_usage,total_tickets,avg_resolution_hours,avg_response_minutes,avg_satisfaction,escalated_tickets
0,A-2e4581,EdTech,United States,Basic,9,0,0,2024-10-16,2024-12-31,2.496715,...,8.0,535.0,152339.0,38.0,4.0,2.0,23.000000,91.000000,3.000000,0.0
1,A-43a9e3,FinTech,India,Basic,18,0,1,2023-08-17,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A-0a282f,DevTools,United States,Basic,1,0,0,2024-08-27,2024-12-31,4.139290,...,15.0,821.0,251210.0,48.0,5.0,3.0,43.666667,63.666667,4.666667,0.0
3,A-1f0ac7,HealthTech,United Kingdom,Basic,24,1,0,2023-08-27,2024-12-31,16.162943,...,5.0,382.0,102528.0,21.0,5.0,2.0,29.000000,174.000000,NaN,0.0
4,A-ce550d,HealthTech,United States,Enterprise,35,0,1,2024-10-27,2024-11-28,1.051248,...,3.0,559.0,209127.0,28.0,4.0,7.0,42.285714,107.857143,3.800000,1.0


In [12]:
target = "churn_flag"

features = [
    "plan_tier",
    "industry",
    "country",
    "seats",
    "is_trial",
    "avg_mrr",
    "upgrade_count",
    "downgrade_count",
    "auto_renew_count",
    "total_usage",
    "total_usage_duration",
    "total_errors",
    "beta_feature_usage",
    "total_tickets",
    "avg_resolution_hours",
    "avg_response_minutes",
    "avg_satisfaction",
    "escalated_tickets",
    "tenure_months"
]

X = model_data[features]
y = model_data[target]

print("Number of features:", len(features))
print("Target distribution:")
print(y.value_counts())

Number of features: 19
Target distribution:
churn_flag
0    390
1    110
Name: count, dtype: int64


In [13]:
categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['plan_tier', 'industry', 'country']

Numerical features:
['seats', 'is_trial', 'avg_mrr', 'upgrade_count', 'downgrade_count', 'auto_renew_count', 'total_usage', 'total_usage_duration', 'total_errors', 'beta_feature_usage', 'total_tickets', 'avg_resolution_hours', 'avg_response_minutes', 'avg_satisfaction', 'escalated_tickets', 'tenure_months']


In [14]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numerical_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training samples: 400
Testing samples: 100

Training target distribution:
churn_flag
0    312
1     88
Name: count, dtype: int64

Testing target distribution:
churn_flag
0    78
1    22
Name: count, dtype: int64


In [16]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (400, 31)
Processed testing shape: (100, 31)


In [17]:
logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_processed,
    y_train
)

print("Balanced Logistic Regression trained.")

Balanced Logistic Regression trained.


c:\Users\Pratham\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [18]:
random_forest_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

print("Balanced Random Forest trained.")

Balanced Random Forest trained.


In [19]:
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=5,
    random_state=42
)

gradient_boosting_model.fit(
    X_train_processed,
    y_train
)

print("Gradient Boosting trained.")

Gradient Boosting trained.


In [20]:
def evaluate_model(model):
    predictions = model.predict(X_test_processed)
    probabilities = model.predict_proba(X_test_processed)[:, 1]

    return {
        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            probabilities
        )
    }

In [21]:
model_results = pd.DataFrame({
    "Logistic Regression": evaluate_model(
        logistic_model
    ),
    "Random Forest": evaluate_model(
        random_forest_model
    ),
    "Gradient Boosting": evaluate_model(
        gradient_boosting_model
    )
}).T

model_results.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.67,0.343,0.545,0.421,0.737
Random Forest,0.89,0.824,0.636,0.718,0.889
Gradient Boosting,0.94,1.000,0.727,0.842,0.864


In [22]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model
}

for name, model in models.items():
    predictions = model.predict(X_test_processed)

    print(f"\n{name}")
    print(confusion_matrix(y_test, predictions))


Logistic Regression
[[55 23]
 [10 12]]

Random Forest
[[75  3]
 [ 8 14]]

Gradient Boosting
[[78  0]
 [ 6 16]]


In [23]:
gb_predictions = gradient_boosting_model.predict(
    X_test_processed
)

print(
    classification_report(
        y_test,
        gb_predictions,
        target_names=["Retained", "Churned"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

    Retained       0.93      1.00      0.96        78
     Churned       1.00      0.73      0.84        22

    accuracy                           0.94       100
   macro avg       0.96      0.86      0.90       100
weighted avg       0.94      0.94      0.94       100



In [24]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        min_samples_leaf=5,
        random_state=42
    ))
])

cv_results = cross_validate(
    gb_pipeline,
    X,
    y,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ]
)

print("Cross-validation results:\n")

for metric in [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]:
    scores = cv_results[f"test_{metric}"]

    print(
        f"{metric.upper():10}: "
        f"{scores.mean():.3f} ± {scores.std():.3f}"
    )

Cross-validation results:

ACCURACY  : 0.926 ± 0.015
PRECISION : 0.924 ± 0.051
RECALL    : 0.727 ± 0.076
F1        : 0.810 ± 0.043
ROC_AUC   : 0.916 ± 0.033


In [25]:
test_results = model_results.loc[
    ["Random Forest", "Gradient Boosting"]
]

print("Test-set results:")
print(test_results.round(3))

print("\nGradient Boosting CV:")
print(
    pd.DataFrame({
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC-AUC"
        ],
        "CV Mean": [
            cv_results["test_accuracy"].mean(),
            cv_results["test_precision"].mean(),
            cv_results["test_recall"].mean(),
            cv_results["test_f1"].mean(),
            cv_results["test_roc_auc"].mean()
        ]
    }).round(3)
)

Test-set results:
                   Accuracy  Precision  Recall     F1  ROC-AUC
Random Forest          0.89      0.824   0.636  0.718    0.889
Gradient Boosting      0.94      1.000   0.727  0.842    0.864

Gradient Boosting CV:
      Metric  CV Mean
0   Accuracy    0.926
1  Precision    0.924
2     Recall    0.727
3         F1    0.810
4    ROC-AUC    0.916


In [26]:
final_model = gb_pipeline

final_model.fit(X, y)

joblib.dump(
    final_model,
    "churn_model.pkl"
)

print("Final Gradient Boosting model saved.")

Final Gradient Boosting model saved.


In [27]:
feature_names = final_model.named_steps[
    "preprocessor"
].get_feature_names_out()

print("Total processed features:", len(feature_names))
print("\nFeature names:")
print(feature_names)

Total processed features: 31

Feature names:
['num__seats' 'num__is_trial' 'num__avg_mrr' 'num__upgrade_count'
 'num__downgrade_count' 'num__auto_renew_count' 'num__total_usage'
 'num__total_usage_duration' 'num__total_errors' 'num__beta_feature_usage'
 'num__total_tickets' 'num__avg_resolution_hours'
 'num__avg_response_minutes' 'num__avg_satisfaction'
 'num__escalated_tickets' 'num__tenure_months' 'cat__plan_tier_Basic'
 'cat__plan_tier_Enterprise' 'cat__plan_tier_Pro'
 'cat__industry_Cybersecurity' 'cat__industry_DevTools'
 'cat__industry_EdTech' 'cat__industry_FinTech' 'cat__industry_HealthTech'
 'cat__country_Australia' 'cat__country_Canada' 'cat__country_France'
 'cat__country_Germany' 'cat__country_India' 'cat__country_United Kingdom'
 'cat__country_United States']


In [28]:
model = final_model.named_steps["model"]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

feature_importance.head(15)

,feature,importance
0,num__avg_mrr,0.341178
1,num__auto_renew_count,0.298205
2,num__tenure_months,0.201608
3,num__total_tickets,0.039857
4,num__total_usage_duration,0.035000
5,num__avg_resolution_hours,0.026438
6,num__avg_response_minutes,0.018347
7,num__seats,0.014194
8,num__avg_satisfaction,0.010590
9,num__total_errors,0.005386


In [29]:
feature_importance["feature"] = (
    feature_importance["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
    .str.replace("_", " ", regex=False)
)

feature_importance["feature"] = (
    feature_importance["feature"]
    .str.title()
)

feature_importance.head(15)

,feature,importance
0,Avg Mrr,0.341178
1,Auto Renew Count,0.298205
2,Tenure Months,0.201608
3,Total Tickets,0.039857
4,Total Usage Duration,0.035000
5,Avg Resolution Hours,0.026438
6,Avg Response Minutes,0.018347
7,Seats,0.014194
8,Avg Satisfaction,0.010590
9,Total Errors,0.005386


In [30]:
top_features = feature_importance.head(10).copy()

print("Top 10 features influencing churn:\n")
print(top_features)

Top 10 features influencing churn:

                feature  importance
0               Avg Mrr    0.341178
1      Auto Renew Count    0.298205
2         Tenure Months    0.201608
3         Total Tickets    0.039857
4  Total Usage Duration    0.035000
5  Avg Resolution Hours    0.026438
6  Avg Response Minutes    0.018347
7                 Seats    0.014194
8      Avg Satisfaction    0.010590
9          Total Errors    0.005386


In [31]:
top_features.to_csv(
    "feature_importance.csv",
    index=False
)

print("Feature importance saved successfully.")

Feature importance saved successfully.


In [32]:
# Predict churn probability for the test set
gb_probabilities = final_model.predict_proba(X_test)[:, 1]

print("First 10 churn probabilities:")
print(
    np.round(gb_probabilities[:10], 3)
)

First 10 churn probabilities:
[0.023 0.13  0.024 0.038 0.02  0.027 0.279 0.154 0.044 0.845]


In [33]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

threshold_results = []

for threshold in thresholds:
    predictions = (
        gb_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results.round(3)

,Threshold,Precision,Recall,F1
0,0.3,1.0,0.864,0.927
1,0.4,1.0,0.773,0.872
2,0.5,1.0,0.773,0.872
3,0.6,1.0,0.773,0.872
4,0.7,1.0,0.727,0.842


In [34]:
best_threshold = threshold_results.loc[
    threshold_results["F1"].idxmax(),
    "Threshold"
]

print("Best threshold:", best_threshold)

Best threshold: 0.3


In [35]:
final_test_predictions = (
    gb_probabilities >= best_threshold
).astype(int)

final_metrics = {
    "Accuracy": accuracy_score(
        y_test,
        final_test_predictions
    ),
    "Precision": precision_score(
        y_test,
        final_test_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        final_test_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        final_test_predictions,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        gb_probabilities
    )
}

print("Final test performance:\n")

for metric, value in final_metrics.items():
    print(f"{metric}: {value:.3f}")

Final test performance:

Accuracy: 0.970
Precision: 1.000
Recall: 0.864
F1: 0.927
ROC-AUC: 0.988


In [36]:
joblib.dump(
    best_threshold,
    "churn_threshold.pkl"
)

print("Prediction threshold saved successfully.")

Prediction threshold saved successfully.


In [37]:
print("final_model exists:", "final_model" in globals())

if "final_model" in globals():
    print("Model type:", type(final_model))
    print("Pipeline steps:", final_model.named_steps.keys() if hasattr(final_model, "named_steps") else "Not a pipeline")

final_model exists: True
Model type: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: dict_keys(['preprocessor', 'model'])


In [38]:
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

model_dir = project_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, model_dir / "churn_model.pkl")

print("Model saved:", model_dir / "churn_model.pkl")
print("Model type:", type(final_model))

Model saved: c:\Users\Pratham\OneDrive\Desktop\Saas_project_details\models\churn_model.pkl
Model type: <class 'sklearn.pipeline.Pipeline'>


In [39]:
loaded_model = joblib.load(model_dir / "churn_model.pkl")

print("Model loaded successfully!")
print("Pipeline steps:", loaded_model.named_steps.keys())

Model loaded successfully!
Pipeline steps: dict_keys(['preprocessor', 'model'])
